In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # tolerate duplicate OpenMP libs
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
torch.set_num_threads(1)
print(torch.__version__)

2.2.2


In [4]:
import pandas as pd
from Bio.PDB import PDBParser

print("Pandas:", pd.__version__)
print("Biopython is working!")

Pandas: 3.0.5
Biopython is working!


In [5]:
import pandas as pd 
import numpy as np 

In [6]:
df = pd.read_csv("skempi_v2.csv", sep=";")
print(df.shape)
print(df.columns.tolist())
df[["#Pdb", "Mutation(s)_cleaned", "Affinity_mut_parsed",
    "Affinity_wt_parsed", "Temperature", "iMutation_Location(s)"]].head()


(7085, 29)
['#Pdb', 'Mutation(s)_PDB', 'Mutation(s)_cleaned', 'iMutation_Location(s)', 'Hold_out_type', 'Hold_out_proteins', 'Affinity_mut (M)', 'Affinity_mut_parsed', 'Affinity_wt (M)', 'Affinity_wt_parsed', 'Reference', 'Protein 1', 'Protein 2', 'Temperature', 'kon_mut (M^(-1)s^(-1))', 'kon_mut_parsed', 'kon_wt (M^(-1)s^(-1))', 'kon_wt_parsed', 'koff_mut (s^(-1))', 'koff_mut_parsed', 'koff_wt (s^(-1))', 'koff_wt_parsed', 'dH_mut (kcal mol^(-1))', 'dH_wt (kcal mol^(-1))', 'dS_mut (cal mol^(-1) K^(-1))', 'dS_wt (cal mol^(-1) K^(-1))', 'Notes', 'Method', 'SKEMPI version,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,']


,#Pdb,Mutation(s)_cleaned,Affinity_mut_parsed,Affinity_wt_parsed,Temperature,iMutation_Location(s)
0,1CSE_E_I,LI38G,5.260000e-11,1.120000e-12,294,COR
1,1CSE_E_I,LI38S,8.330000e-12,1.120000e-12,294,COR
2,1CSE_E_I,LI38P,1.020000e-07,1.120000e-12,294,COR
3,1CSE_E_I,LI38I,1.720000e-10,1.120000e-12,294,COR
4,1CSE_E_I,LI38D,1.920000e-09,1.120000e-12,294,COR


In [ ]:
df = df[~df["Mutation(s)_cleaned"].str.contains(",", na=False)].copy()
print(df.shape)

# filtering the data to only include single-point mutations, found by searching the "Mutation(s)_cleaned" column for entries that do not contain a comma

(5112, 29)


In [ ]:
cols = ["Affinity_mut_parsed", "Affinity_wt_parsed", "Temperature"]
print(df[cols].isna().sum())  # how many are missing per column

df = df.dropna(subset=cols)
df = df[(df["Affinity_mut_parsed"] > 0) & (df["Affinity_wt_parsed"] > 0)]
print(df.shape)

# filtering the data to only include rows where there are no missing values in affinity_mut, affinity_wt, and temp. also filters out rows where affinity_mut and affinity_wt are negative. 

Affinity_mut_parsed    154
Affinity_wt_parsed       2
Temperature              4
dtype: int64
(4952, 29)


In [ ]:
df["Temperature"] = (
    df["Temperature"].astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float)
)
# converts temperature column to float, however, the calculations assume that the temperature is in Kelvin. 

R = 0.001987  # kcal/(mol*K)
df["ddG"] = R * df["Temperature"] * np.log(
    df["Affinity_mut_parsed"] / df["Affinity_wt_parsed"]
) # a (+ve) ddG means the mutation is destabilizing, a (-ve) ddG means the mutation is stabilizing
df["ddG"].describe()

count    4952.000000
mean        1.002189
std         1.727207
min       -12.221147
25%         0.012891
50%         0.598058
75%         1.720191
max        12.221147
Name: ddG, dtype: float64

In [ ]:
import re

pat = re.compile(r"^([A-Z])([A-Za-z])(-?\d+)([A-Za-z]?)([A-Z])$")

def parse(m):
    g = pat.match(m)
    if not g:
        return pd.Series([None]*5)
    wt, chain, num, icode, mt = g.groups()
    return pd.Series([wt, chain, int(num), icode, mt])

df[["wt", "chain", "pos", "icode", "mt"]] = df["Mutation(s)_cleaned"].apply(parse)
print(df["wt"].isna().sum(), "rows failed to parse")
df = df.dropna(subset=["wt"])

# use regex expressions to split the mut_cleaned column into 5 new columns: wt, chain, pos, icode, and mut

0 rows failed to parse


In [ ]:
df["label_binary"] = (df["ddG"] > 0).astype(int)   # 1 = destabilizing
print(df["label_binary"].value_counts(normalize=True))

# creating a new column which tells which mutations are destabilizing and gives it a binary label of 1, and then which are stabilizing, giving them a binary label of 0. 

label_binary
1    0.752221
0    0.247779
Name: proportion, dtype: float64


In [12]:
dup = df.duplicated(subset=["#Pdb", "Mutation(s)_cleaned"], keep=False).sum()
print(dup, "rows share the same complex + mutation")

df.to_csv("skempi_single_clean.csv", index=False)
print(df.shape)

1260 rows share the same complex + mutation
(4952, 36)


In [13]:
cols = ["#Pdb", "Mutation(s)_cleaned", "Affinity_mut_parsed",
        "Affinity_wt_parsed", "Temperature", "ddG"]
srt = df.sort_values("ddG")
print(srt.head(6)[cols])
print(srt.tail(6)[cols])

          #Pdb Mutation(s)_cleaned  Affinity_mut_parsed  Affinity_wt_parsed  \
3137  3BTG_E_I               GI13K         5.880000e-14        6.670000e-05   
3134  3BTD_E_I               DI13K         5.880000e-14        1.590000e-05   
3142  3BTT_E_I               TI13K         5.880000e-14        3.450000e-06   
3135  3BTE_E_I               EI13K         5.880000e-14        4.760000e-07   
3140  3BTQ_E_I               QI13K         5.880000e-14        4.350000e-07   
3138  3BTH_E_I               HI13K         5.880000e-14        1.610000e-07   

      Temperature        ddG  
3137        295.0 -12.221147  
3134        295.0 -11.380653  
3142        295.0 -10.485025  
3135        295.0  -9.324002  
3140        295.0  -9.271205  
3138        295.0  -8.688591  
          #Pdb Mutation(s)_cleaned  Affinity_mut_parsed  Affinity_wt_parsed  \
1346  2FTL_E_I               KI15T             0.000003        5.880000e-14   
1343  2FTL_E_I               KI15A             0.000004        5.880000

In [ ]:
key = ["#Pdb", "Mutation(s)_cleaned"]
spread = df.groupby(key)["ddG"].agg(["count", "min", "max"])
spread = spread[spread["count"] > 1].copy()
# create a new df which only has duplicated ddG values.
spread["range"] = spread["max"] - spread["min"]

print(len(spread), "duplicated mutation keys")
print(spread["range"].describe())
print((spread["range"] > 1).sum(), "keys with range > 1 kcal/mol")
print(((spread["min"] < 0) & (spread["max"] > 0)).sum(), "keys with conflicting signs")
# mutations where a ddG can be (-)ve and (+)ve 

499 duplicated mutation keys
count    499.000000
mean       0.485170
std        0.607756
min        0.000000
25%        0.094742
50%        0.299151
75%        0.619496
max        5.364922
Name: range, dtype: float64
75 keys with range > 1 kcal/mol
48 keys with conflicting signs


In [15]:
med = df.groupby(key)["ddG"].median()
conf = spread[(spread["min"] < 0) & (spread["max"] > 0)].copy()
conf["abs_median"] = med.loc[conf.index].abs()

print(conf["abs_median"].describe())
print((conf["abs_median"] < 0.5).sum(), "of", len(conf),
      "conflicting keys have |median ddG| < 0.5")

count    48.000000
mean      0.359380
std       0.541064
min       0.000000
25%       0.046723
50%       0.139766
75%       0.385342
max       2.093005
Name: abs_median, dtype: float64
39 of 48 conflicting keys have |median ddG| < 0.5


In [ ]:
agg = df.groupby(key).agg(
    ddG=("ddG", "median"),
    n_meas=("ddG", "size"),
    ddG_min=("ddG", "min"),
    ddG_max=("ddG", "max"),
    wt=("wt", "first"), chain=("chain", "first"),
    pos=("pos", "first"), icode=("icode", "first"), mt=("mt", "first"),
    mut_location=("iMutation_Location(s)", "first"),
).reset_index()

agg["ambiguous"] = (
    ((agg["ddG_max"] - agg["ddG_min"]) > 1)
    | ((agg["ddG_min"] < 0) & (agg["ddG_max"] > 0))
)
# creating a new column called ambiguous which has 2 conditions: 1) if range > 1 kcal/mol, or 2) if there are conflicting ddGs
agg["label_binary"] = (agg["ddG"] > 0).astype(int)

print(agg.shape)
print(agg["ambiguous"].sum(), "ambiguous rows")
print(agg["label_binary"].value_counts(normalize=True))
agg.to_csv("skempi_single_collapsed.csv", index=False)

# creating a cleaned and collapsed df. 

(4191, 14)
107 ambiguous rows
label_binary
1    0.751849
0    0.248151
Name: proportion, dtype: float64


In [ ]:
import pandas as pd
from Bio.PDB import PDBParser

AA3 = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
       "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
       "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V"}

parser = PDBParser(QUIET=True)
cache = {}

def chain_info(pdb_id, chain):
    # inputs = pdb id, and chain, outputs = aa sequence and positions of the residues in the sequence
    k = (pdb_id, chain)
    if k not in cache:
        model = parser.get_structure(pdb_id, f"PDBs/{pdb_id}.pdb")[0]
        seq, idx = [], {}
        for res in model[chain]:
            if res.id[0] != " " or res.get_resname() not in AA3:
                continue  
            # skip waters, ligands, modified residues
            idx[(res.id[1], res.id[2].strip())] = len(seq)
            seq.append(AA3[res.get_resname()])
        cache[k] = ("".join(seq), idx)
    return cache[k]

In [ ]:
agg = pd.read_csv("skempi_single_collapsed.csv")
agg["pdb_id"] = agg["#Pdb"].str.split("_").str[0]
agg["icode"] = agg["icode"].fillna("")

status, seqs, idxs = [], [], []
for _, r in agg.iterrows():
    try:
        seq, idx = chain_info(r["pdb_id"], r["chain"])
        i = idx.get((int(r["pos"]), r["icode"]))
        if i is None:
            status.append("pos_not_found"); seqs.append(None); idxs.append(None)
        elif seq[i] != r["wt"]:
            status.append("wt_mismatch"); seqs.append(None); idxs.append(None)
        else:
            status.append("ok"); seqs.append(seq); idxs.append(i)
    except Exception as e:
        status.append("error"); seqs.append(None); idxs.append(None)

agg["status"], agg["seq"], agg["seq_idx"] = status, seqs, idxs
print(agg["status"].value_counts())
print(agg.loc[agg["status"] != "ok", "pdb_id"].value_counts().head(10))

# ensuring mapping was succeful, on the basis of finding the mutation position, checking that WT matches 

status
ok    4191
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


In [13]:
ok = agg[agg["status"] == "ok"].copy()
ok.to_csv("skempi_with_seq.csv", index=False)
print(ok.shape)
ok

NameError: name 'agg' is not defined

In [ ]:
import torch, pandas as pd
from transformers import AutoTokenizer, EsmForMaskedLM

ok = pd.read_csv("skempi_with_seq.csv")
ok = ok[ok["seq"].str.len() <= 1022].copy()   

name = "facebook/esm2_t12_35M_UR50D"   # small; swap for t33_650M on the cluster GPU
tok = AutoTokenizer.from_pretrained(name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = EsmForMaskedLM.from_pretrained(name).to(device).eval()
print(device, ok.shape)

/opt/miniconda3/envs/esm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/esm/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


cpu (4152, 18)


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, EsmForMaskedLM

ok = pd.read_csv("skempi_with_seq.csv")
ok = ok[ok["seq"].str.len() <= 1022].copy()

name = "facebook/esm2_t12_35M_UR50D"
tok = AutoTokenizer.from_pretrained(name)
device = "cpu"
model = EsmForMaskedLM.from_pretrained(name).to(device).eval()

r = ok.iloc[0]
masked = r["seq"][:int(r["seq_idx"])] + tok.mask_token + r["seq"][int(r["seq_idx"])+1:]
# gives model everything before, masks the mutation token, and gives everything after
enc = tok(masked, return_tensors="pt")
with torch.no_grad():
    out = model(**enc).logits
print(enc["input_ids"].shape, out.shape)
# (1, L), where 1 indicates that 1 sequence was processed, and L is the sequences length (including the special tokens) & (1, L, V), where V is the possible tokens.

/opt/miniconda3/envs/esm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/esm/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


torch.Size([1, 182]) torch.Size([1, 182, 33])


In [ ]:
import time

aa_ids = {a: tok.convert_tokens_to_ids(a) for a in "ACDEFGHIKLMNPQRSTVWY"}
pos_cache = {}

@torch.no_grad()
def position_logp(seq, i):
    key = (seq, i)
    if key not in pos_cache:
        masked = seq[:i] + tok.mask_token + seq[i+1:]
        enc = tok(masked, return_tensors="pt")
        logits = model(**enc).logits[0, i + 1]
        pos_cache[key] = torch.log_softmax(logits, dim=-1).clone()
    return pos_cache[key]
# running the ESM, and converting logits into log probabilities 

def llr(r):
    lp = position_logp(r["seq"], int(r["seq_idx"]))
    return (lp[aa_ids[r["mt"]]] - lp[aa_ids[r["wt"]]]).item()
# calculates llr 

test = ok.head(20).copy()
t0 = time.time()
test["llr"] = test.apply(llr, axis=1)
print(time.time() - t0, "seconds for 20 rows")
print(test["llr"].describe())

6.3992600440979 seconds for 20 rows
count    20.000000
mean     -0.875680
std       0.976605
min      -3.017035
25%      -1.453723
50%      -0.673657
75%      -0.326452
max       1.434426
Name: llr, dtype: float64


In [4]:
import numpy as np

ok["llr"] = np.nan
for start in range(0, len(ok), 250):
    chunk = ok.iloc[start:start+250]
    ok.loc[chunk.index, "llr"] = chunk.apply(llr, axis=1)
    ok.to_csv("skempi_scored_partial.csv", index=False)
    print(start + len(chunk), "/", len(ok), flush=True)

250 / 4152
500 / 4152
750 / 4152
1000 / 4152
1250 / 4152
1500 / 4152
1750 / 4152
2000 / 4152
2250 / 4152
2500 / 4152
2750 / 4152
3000 / 4152
3250 / 4152
3500 / 4152
3750 / 4152
4000 / 4152
4152 / 4152


In [5]:
print(ok["llr"].isna().sum(), "rows with no score")
print(ok["llr"].describe())
print(ok.groupby("label_binary")["llr"].mean())
ok.to_csv("skempi_scored.csv", index=False)

0 rows with no score
count    4152.000000
mean       -0.456048
std         1.380604
min       -12.902997
25%        -0.992238
50%        -0.264828
75%         0.314282
max         7.244984
Name: llr, dtype: float64
label_binary
0   -0.519670
1   -0.435112
Name: llr, dtype: float64


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
from scipy.stats import spearmanr

def evaluate(d, name):
    y, s = d["label_binary"], -d["llr"]          
    # higher = more likely destabilizing
    # y = experimental binary labels, s = esm scores ((-)ve LLR = mutant less plausible)
    return {
        "subset": name, "n": len(d), "pct_destab": y.mean(),
        # mean is fraction of rows that have a label of 1, meaning destabilizing 
        "AUROC": roc_auc_score(y, s),
        "AUPRC": average_precision_score(y, s),
        "Spearman": spearmanr(s, d["ddG"])[0],
        "MCC@0": matthews_corrcoef(y, (d["llr"] < 0).astype(int)),
    }

subsets = {
    "all": ok,
    "non-ambiguous": ok[~ok["ambiguous"]],
    "|ddG| >= 0.5": ok[ok["ddG"].abs() >= 0.5],
}
# there are 3 subsets; all is for evrything (overall performance), non-ambiguous is for when you exclude ambiguous rows, and |ddG| >= 0.5 is for mutations with large effects. 
results = pd.DataFrame([evaluate(d, n) for n, d in subsets.items()])
results.round(3)

,subset,n,pct_destab,AUROC,AUPRC,Spearman,MCC@0
0,all,4152,0.752,0.459,0.738,-0.045,-0.068
1,non-ambiguous,4046,0.752,0.461,0.738,-0.040,-0.066
2,|ddG| >= 0.5,2502,0.867,0.467,0.862,0.007,-0.040


In [ ]:
print(ok["mut_location"].value_counts())

loc = (ok.groupby("mut_location")
         .filter(lambda d: len(d) >= 50 and d["label_binary"].nunique() == 2)
         # each group must have at least 50 rows, and must have both destabilizing and stabilizing mutations
         .groupby("mut_location")
         .apply(lambda d: pd.Series(evaluate(d, d.name))))
print(loc[["n", "pct_destab", "AUROC", "Spearman"]].round(3))

ok["pdb_id"] = ok["#Pdb"].str.split("_").str[0]
top = ok["pdb_id"].value_counts().head(10)
print(top)

# evaluates performance based on the location of the mutation 

mut_location
COR    1756
RIM     984
SUR     563
SUP     523
INT     326
Name: count, dtype: int64
                 n  pct_destab  AUROC  Spearman
mut_location                                   
COR           1756       0.820  0.466     0.027
INT            326       0.666  0.505     0.072
RIM            984       0.704  0.441    -0.089
SUP            523       0.851  0.531    -0.003
SUR            563       0.584  0.468    -0.064
pdb_id
3BT1    240
1CHO    191
1R0R    191
3SGB    191
1PPF    190
1A22    142
1DAN     89
1AO7     86
3S9D     81
3HFM     71
Name: count, dtype: int64


In [ ]:
ok["mut_better"] = (ok["ddG"] < 0).astype(int)   
# creation of a binary label, where 1 = ddG < 0, mutant is stabilizing, and 0 = ddG > 0, mutant is destabilizing.
valid = ok[(ok["llr"] != 0) & (ok["ddG"] != 0)].copy()   # ties have no direction
# valid is a new df, which has ESM scores, and ddG values that are not 0 
valid["mut_more_probable"] = (valid["llr"] > 0).astype(int)
# creation of a binary label, where 1 = (+)ve LLR, 0 = (-)ve LLR
valid["agree"] = (valid["mut_more_probable"] == valid["mut_better"]).astype(int)

print(len(ok) - len(valid), "rows dropped (exact ties)")
print("overall agreement:", round(valid["agree"].mean(), 3))
print(valid.groupby("mut_better")["agree"].agg(["mean", "size"]).round(3))

142 rows dropped (exact ties)
overall agreement: 0.541
             mean  size
mut_better             
0           0.605  3124
1           0.316   886


In [ ]:
p_wt_more_probable = (valid["llr"] < 0).mean()
# how often does ESM prefer WT aa
p_wt_better = (valid["mut_better"] == 0).mean()
# how often does WT aa bind better than mutant aa
baseline = (p_wt_more_probable * p_wt_better
            + (1 - p_wt_more_probable) * (1 - p_wt_better))
# agreement expected by chance, adds up 2 possibilites (favouring WT + favouring mut)

balanced = valid.groupby("mut_better")["agree"].mean().mean()
# balanced agreement 

print("share where wt is more probable:", round(p_wt_more_probable, 3))
print("share where wt binds better:   ", round(p_wt_better, 3))
print("agreement expected by chance:  ", round(baseline, 3))
print("balanced agreement (0.5 = chance):", round(balanced, 3))

share where wt is more probable: 0.622
share where wt binds better:    0.779
agreement expected by chance:   0.568
balanced agreement (0.5 = chance): 0.461
